# AutoPilot Headless Kaggle API


In [ ]:
!pip install torch==2.6.0 torchvision==0.21.0
!pip install -q torchsde einops diffusers accelerate xformers==0.0.29.post2 av fastapi uvicorn pyngrok pydantic imageio imageio-ffmpeg python-multipart
!apt-get -y install -qq aria2 ffmpeg

%cd /kaggle/working
!git clone https://github.com/Isi-dev/ComfyUI
%cd /kaggle/working/ComfyUI/custom_nodes
!git clone https://github.com/Isi-dev/ComfyUI_GGUF.git
%cd /kaggle/working/ComfyUI/custom_nodes/ComfyUI_GGUF
!pip install -r requirements.txt
%cd /kaggle/working/ComfyUI

# Download Models
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/city96/Wan2.1-I2V-14B-480P-gguf/resolve/main/wan2.1-i2v-14b-480p-Q6_K.gguf -d /kaggle/working/ComfyUI/models/unet -o wan2.1-i2v-14b-480p-Q6_K.gguf
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors -d /kaggle/working/ComfyUI/models/text_encoders -o umt5_xxl_fp8_e4m3fn_scaled.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors -d /kaggle/working/ComfyUI/models/vae -o wan_2.1_vae.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors -d /kaggle/working/ComfyUI/models/clip_vision -o clip_vision_h.safetensors



In [ ]:
import torch
import numpy as np
from PIL import Image
import gc
import sys
import random
import os
import imageio
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel
import uvicorn
from pyngrok import ngrok
import asyncio

sys.path.insert(0, '/kaggle/working/ComfyUI')

from comfy import model_management

from nodes import (
    CheckpointLoaderSimple,
    CLIPLoader,
    CLIPTextEncode,
    VAEDecode,
    VAELoader,
    KSampler,
    UNETLoader,
    LoadImage,
    CLIPVisionLoader,
    CLIPVisionEncode
)

from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
from comfy_extras.nodes_model_advanced import ModelSamplingSD3
from comfy_extras.nodes_images import SaveAnimatedWEBP
from comfy_extras.nodes_video import SaveWEBM
from comfy_extras.nodes_wan import WanImageToVideo

# Initialize nodes globally
unet_loader = UnetLoaderGGUF()
model_sampling = ModelSamplingSD3()
clip_loader = CLIPLoader()
clip_encode_positive = CLIPTextEncode()
clip_encode_negative = CLIPTextEncode()
vae_loader = VAELoader()
clip_vision_loader = CLIPVisionLoader()
clip_vision_encode = CLIPVisionEncode()
load_image_node = LoadImage()
wan_image_to_video = WanImageToVideo()
ksampler = KSampler()
vae_decode = VAEDecode()

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    for obj in list(globals().values()):
        if torch.is_tensor(obj) or (hasattr(obj, "data") and torch.is_tensor(obj.data)):
            del obj
    gc.collect()

def save_as_mp4(images, filename_prefix, fps, output_dir="/kaggle/working/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.mp4"
    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]
    with imageio.get_writer(output_path, fps=fps) as writer:
        for frame in frames:
            writer.append_data(frame)
    return output_path

def generate_video(
    image_path: str,
    positive_prompt: str,
    negative_prompt: str = "text, watermark, ugly, worst quality, low quality",
    width: int = 480,
    height: int = 832,
    seed: int = 0,
    steps: int = 20,
    cfg_scale: float = 3.0,
    sampler_name: str = "uni_pc",
    scheduler: str = "simple",
    frames: int = 49,
    fps: int = 16
):
    if seed == 0:
        seed = random.randint(0, 2**32 - 1)
        
    with torch.inference_mode():
        print("Loading Text_Encoder...")
        clip = clip_loader.load_clip("umt5_xxl_fp8_e4m3fn_scaled.safetensors", "wan", "default")[0]
        positive = clip_encode_positive.encode(clip, positive_prompt)[0]
        negative = clip_encode_negative.encode(clip, negative_prompt)[0]

        del clip
        clear_memory()

        print(f"Loading Image: {image_path}")
        loaded_image = load_image_node.load_image(image_path)[0]
        clip_vision = clip_vision_loader.load_clip("clip_vision_h.safetensors")[0]
        clip_vision_output = clip_vision_encode.encode(clip_vision, loaded_image, "none")[0]

        del clip_vision
        clear_memory()

        print("Loading VAE...")
        vae = vae_loader.load_vae("wan_2.1_vae.safetensors")[0]

        positive_out, negative_out, latent = wan_image_to_video.encode(
            positive, negative, vae, width, height, frames, 1, loaded_image, clip_vision_output
        )

        print("Loading Unet Model...")
        model = unet_loader.load_unet("wan2.1-i2v-14b-480p-Q6_K.gguf")[0]
        model = model_sampling.patch(model, 8)[0]

        print("Generating video...")
        sampled = ksampler.sample(
            model=model,
            seed=seed,
            steps=steps,
            cfg=cfg_scale,
            sampler_name=sampler_name,
            scheduler=scheduler,
            positive=positive_out,
            negative=negative_out,
            latent_image=latent
        )[0]

        del model
        clear_memory()

        print("Decoding latents...")
        decoded = vae_decode.decode(vae, sampled)[0]
        del vae
        clear_memory()

        output_path = save_as_mp4(decoded, f"wan_video_{seed}", fps)
        return output_path

# --- FastAPI Server ---
app = FastAPI()

@app.post("/generate_video")
async def api_generate_video(
    image: UploadFile = File(...),
    prompt: str = Form(...),
    seed: int = Form(0),
    steps: int = Form(20)
):
    try:
        os.makedirs("/kaggle/working/input", exist_ok=True)
        image_path = f"/kaggle/working/input/{image.filename}"
        with open(image_path, "wb") as buffer:
            buffer.write(await image.read())
            
        print(f"Received request: {prompt}")
        output_mp4 = generate_video(
            image_path=image_path,
            positive_prompt=prompt,
            seed=seed,
            steps=steps
        )
        return FileResponse(output_mp4, media_type="video/mp4", filename="output.mp4")
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    # Get Ngrok Authtoken
    NGROK_AUTH_TOKEN = "your_ngrok_token_here"  # YOU MUST ENTER YOUR NGROK TOKEN HERE
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(8000)
    print(f"\n\n>>> FASTAPI URL: {public_url.public_url} <<<\n\n")
    
    uvicorn.run(app, host="0.0.0.0", port=8000)

